In [1]:
import time
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import SparseVector, Prefetch

In [27]:
QDRANT_URL = "http://localhost:6333"
QDRANT_COLLECTION = "msmarco_english_corpus"   # must match your existing collection name
EMBED_MODEL = "BAAI/bge-small-en-v1.5"          # must match what was used to build the collection
SPARSE_MODEL = "Qdrant/bm25"

DENSE_WEIGHT = 0.75
SPARSE_WEIGHT = 0.25
TOP_K = 5
CANDIDATE_POOL = 20   # how many candidates each retriever pulls before fusion

In [28]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

client = QdrantClient(url=QDRANT_URL)

dense_model = SentenceTransformer(
    EMBED_MODEL,
    local_files_only=True
)

sparse_model = SparseTextEmbedding(
    model_name=SPARSE_MODEL,
    cache_dir="../hf_cache",   # same path you downloaded to earlier
    local_files_only=True
)

# sanity check: confirm collection exists and has both vector types
info = client.get_collection(QDRANT_COLLECTION)
print("Dense vectors:", info.config.params.vectors)
print("Sparse vectors:", info.config.params.sparse_vectors)
print(f"Total points: {info.points_count}")

Dense vectors: {'dense': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None)}
Sparse vectors: {'sparse': SparseVectorParams(index=None, modifier=None)}
Total points: 29904


In [29]:
def retrieve_hybrid(query_text: str, top_k: int = TOP_K,
                     dense_weight: float = DENSE_WEIGHT, sparse_weight: float = SPARSE_WEIGHT,
                     candidate_pool: int = CANDIDATE_POOL):
    """
    Hybrid retrieval (dense=0.75 / sparse=0.25 weighted fusion) against an EXISTING Qdrant collection.
    Returns a dict with retrieved chunks (ready for LLM context) + timing breakdown.
    """
    t0 = time.perf_counter()

    prefixed = f"Represent this sentence for searching relevant passages: {query_text}"
    dvec = dense_model.encode(prefixed, normalize_embeddings=True).tolist()
    svec = list(sparse_model.embed([query_text]))[0]
    t1 = time.perf_counter()

    dense_resp = client.query_points(
        collection_name=QDRANT_COLLECTION, query=dvec, using="dense", limit=candidate_pool
    )
    sparse_resp = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=SparseVector(indices=svec.indices.tolist(), values=svec.values.tolist()),
        using="sparse", limit=candidate_pool
    )
    t2 = time.perf_counter()

    def normalize(points):
        scores = [p.score for p in points]
        if not scores:
            return {}
        lo, hi = min(scores), max(scores)
        rng = (hi - lo) or 1.0
        return {p.payload.get("passage_id"): (p.score - lo) / rng for p in points}

    dense_norm = normalize(dense_resp.points)
    sparse_norm = normalize(sparse_resp.points)

    point_lookup = {p.payload.get("passage_id"): p for p in dense_resp.points}
    for p in sparse_resp.points:
        point_lookup.setdefault(p.payload.get("passage_id"), p)

    all_ids = set(dense_norm) | set(sparse_norm)
    fused_scores = {
        pid: dense_weight * dense_norm.get(pid, 0.0) + sparse_weight * sparse_norm.get(pid, 0.0)
        for pid in all_ids
    }
    ranked_ids = sorted(fused_scores, key=lambda pid: fused_scores[pid], reverse=True)[:top_k]

    chunks = []
    for pid in ranked_ids:
        p = point_lookup[pid]
        chunks.append({
            "passage_id": pid,
            "text": p.payload.get("text_en", ""),
            "fused_score": round(fused_scores[pid], 4),
            "chunk_strategy": p.payload.get("chunk_strategy"),
            "query_type": p.payload.get("query_type"),
        })

    t3 = time.perf_counter()

    return {
        "query": query_text,
        "chunks": chunks,
        "top1_score": chunks[0]["fused_score"] if chunks else None,
        "embedding_ms": round((t1 - t0) * 1000, 2),
        "qdrant_ms": round((t2 - t1) * 1000, 2),
        "fusion_ms": round((t3 - t2) * 1000, 2),
        "total_ms": round((t3 - t0) * 1000, 2),
    }

In [30]:
result = retrieve_hybrid("what is valgrind")

print(f"Query: {result['query']}")
print(f"Total time: {result['total_ms']}ms  (embed={result['embedding_ms']}ms, qdrant={result['qdrant_ms']}ms, fusion={result['fusion_ms']}ms)")
print(f"\nTop {len(result['chunks'])} chunks:")
for i, c in enumerate(result["chunks"], 1):
    print(f"\n{i}. [{c['passage_id']}] score={c['fused_score']}")
    print(f"   {c['text'][:200]}")

Query: what is valgrind
Total time: 219.24ms  (embed=118.85ms, qdrant=100.22ms, fusion=0.16ms)

Top 5 chunks:

1. [hi_860160_2] score=0.8763
   About Valgrind. Valgrind is a GPL'd system for debugging and profiling Linux programs. With Valgrind's tool suite you can automatically detect many memory management and threading bugs, avoiding hours

2. [hi_860160_6] score=0.864
   Current release: valgrind-3.11.0. Valgrind is an instrumentation framework for building dynamic analysis tools. There are Valgrind tools that can automatically detect many memory management and thread

3. [hi_860160_0] score=0.7755
   Valgrind /ˈvaelɡrɪnd/ ˈvælɡrɪnd is a programming tool for memory, debugging memory leak, detection and. Profiling valgrind was originally designed to be a free memory debugging tool For linux on, x86 

4. [hi_860160_4] score=0.7657
   Valgrind (downloadable here) is a utility for debugging programs for the x86 and x86-64 Linux platforms. It has recently become highly popular as it can

In [31]:
def build_context_for_llm(retrieval_result, max_chunks=5):
    """Formats retrieved chunks into a context string ready to inject into an LLM prompt."""
    context_parts = []
    for i, c in enumerate(retrieval_result["chunks"][:max_chunks], 1):
        context_parts.append(f"[Source {i}]\n{c['text']}")
    return "\n\n".join(context_parts)

# example
context = build_context_for_llm(result)
print(context)

[Source 1]
About Valgrind. Valgrind is a GPL'd system for debugging and profiling Linux programs. With Valgrind's tool suite you can automatically detect many memory management and threading bugs, avoiding hours of frustrating bug-hunting, making your programs more stable.You can also perform detailed profiling to help speed up your programs. Valgrind is not a toy. 2 Valgrind is first and foremost a debugging and profiling system for large, complex programs. 3 We have had feedback from users working on projects with up to 25 million lines of code.

[Source 2]
Current release: valgrind-3.11.0. Valgrind is an instrumentation framework for building dynamic analysis tools. There are Valgrind tools that can automatically detect many memory management and threading bugs, and profile your programs in detail.You can also use Valgrind to build new tools.The Valgrind distribution currently includes six production-quality tools: a memory error detector, two thread error detectors, a cache and bra

# Gemnini  Work

In [9]:
!pip install google-generativeai -q


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import google.generativeai as genai

GEMINI_API_KEY =   # get from https://aistudio.google.com/apikey
genai.configure(api_key=GEMINI_API_KEY)

GEMINI_MODEL = "gemini-3.1-flash-lite"   # "lite" version — fast, cheap, good for latency-sensitive RAG

model_llm = genai.GenerativeModel(GEMINI_MODEL)

In [33]:
MIN_CONFIDENCE_THRESHOLD = 0.55   # tune this based on your benchmark score distributions

def check_retrieval_confidence(retrieval_result):
    """
    Returns (should_proceed: bool, reason: str)
    Blocks generation if retrieval confidence is too low to trust.
    """
    if not retrieval_result["chunks"]:
        return False, "No relevant passages found in the database."

    top1_score = retrieval_result["top1_score"]
    if top1_score < MIN_CONFIDENCE_THRESHOLD:
        return False, f"Low retrieval confidence ({top1_score:.3f} < {MIN_CONFIDENCE_THRESHOLD}). Answer may not be reliable."

    return True, "OK"

In [34]:
def build_prompt_voice(question: str, context: str) -> str:
    return f"""You are a helpful voice assistant. Answer the question using ONLY the information in the context below.

Rules:
- If the context contains relevant information that lets you construct a reasonable answer — even if it's not phrased as an exact definition — synthesize the best answer you can from it.
- Only respond with "I don't have enough information to answer that." if the context is genuinely unrelated to the question or contains nothing useful.
- Do not use any outside knowledge beyond what's in the context.
- Write the answer as natural SPOKEN language — the way a person would say it out loud.
- No bullet points, no markdown, no headers, no numbered lists, no special characters.
- Keep it short: 1-3 sentences, easy to listen to.
- Do not say "according to the context" or reference sources.

Context:
{context}

Question: {question}

Spoken answer:"""

In [14]:
# import time
#
# def generate_answer(question: str, context: str, max_retries: int = 2):
#     prompt = build_prompt_voice(question, context)
#     print(prompt)
#
#     for attempt in range(max_retries + 1):
#         try:
#             t0 = time.perf_counter()
#             response = model_llm.generate_content(
#                 prompt,
#                 generation_config=genai.types.GenerationConfig(
#                     temperature=0.1,       # low temp = more grounded, less creative drift
#                     max_output_tokens=200,
#                 )
#             )
#             t1 = time.perf_counter()
#
#             return {
#                 "answer": response.text.strip(),
#                 "generation_ms": round((t1 - t0) * 1000, 2),
#                 "success": True,
#                 "error": None,
#             }
#         except Exception as e:
#             if attempt == max_retries:
#                 return {
#                     "answer": None,
#                     "generation_ms": None,
#                     "success": False,
#                     "error": str(e),
#                 }
#             time.sleep(0.5 * (attempt + 1))   # simple backoff before retry

# Groq setting

In [81]:
!pip install openai


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# import os
# import time
# import re
# from groq import Groq
#
# # Initialize client outside the function to avoid connection overhead per call
# groq_client = Groq(api_key=)
#
# MODEL_NAME = "allam-2-7b"
#
# def generate_answer(question: str, context: str, max_retries: int = 1):
#     clean_context = context[:500] if context else ""
#
#     messages = [
#         {
#             "role": "system",
#             "content": (
#                 "You are an ultra-fast retrieval engine. "
#                 "Output ONLY a single concise sentence answering the question from the context. "
#                 "Do not add commentary, extra details, or introductory phrases."
#             ),
#         },
#         {
#             "role": "user",
#             "content": f"Context:\n{clean_context}\n\nQuestion:\n{question}",
#         },
#     ]
#
#     for attempt in range(max_retries + 1):
#         try:
#             t0 = time.perf_counter()
#             response = groq_client.chat.completions.create(
#                 model=MODEL_NAME,
#                 messages=messages,
#                 temperature=0.0,
#                 max_tokens=55,      # Enough space for a complete 1-sentence answer without truncation
#                 stream=False
#             )
#             t1 = time.perf_counter()
#
#             content = response.choices[0].message.content or ""
#
#             # Check LPU server processing time vs network round trip
#             lpu_ms = getattr(response.usage, "total_time", 0) * 1000
#
#             return {
#                 "answer": content.strip(),
#                 "generation_ms": round((t1 - t0) * 1000, 2),
#                 "lpu_server_ms": round(lpu_ms, 2),
#                 "success": True,
#                 "error": None,
#             }
#
#         except Exception as e:
#             if attempt == max_retries:
#                 return {
#                     "answer": None,
#                     "generation_ms": None,
#                     "success": False,
#                     "error": str(e),
#                 }
#             time.sleep(0.02)

In [98]:
import time
from openai import OpenAI

# Connect directly to your local Ollama server running on D: drive
ollama_client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama"
)

# Best ultra-fast model for CPU-only retrieval extraction
MODEL_NAME = "qwen2.5:1.5b"

def generate_answer(question: str, context: str, max_retries: int = 1):
    clean_context = context[:500] if context else ""

    messages = [
        {
            "role": "system",
            "content": (
                "You are an ultra-fast retrieval extraction engine. "
                "Output the answer in exactly 3 separate lines, totaling 30 to 40 words. "
                "Line 1: Direct definition.\n"
                "Line 2: Primary capabilities/bugs detected.\n"
                "Line 3: Benefit to software reliability.\n"
                "Do not include intro phrases, bullet labels, or extra text."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{clean_context}\n\nQuestion:\n{question}",
        },
    ]

    for attempt in range(max_retries + 1):
        try:
            t0 = time.perf_counter()
            response = ollama_client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0.0,
                max_tokens=100,      # Set to 80 to ensure 35-40 words are never cut off
                stream=False,
                extra_body={
                    "keep_alive": "10m",
                    "options": {
                        "num_ctx": 512,
                        "num_predict": 20,
                    }
                }
            )
            t1 = time.perf_counter()

            content = response.choices[0].message.content or ""

            return {
                "answer": content.strip(),
                "generation_ms": round((t1 - t0) * 1000, 2),
                "success": True,
                "error": None,
            }

        except Exception as e:
            if attempt == max_retries:
                return {
                    "answer": None,
                    "generation_ms": None,
                    "success": False,
                    "error": str(e),
                }
            time.sleep(0.02)

In [131]:
import time
import requests

url = "http://127.0.0.1:11434/api/chat"

payload = {
    "model": "qwen2.5:0.5b",
    "messages": [
        {
            "role": "system",
            "content": "Answer only from the context. Be concise."
        },
        {
            "role": "user",
            "content": """Context:
Valgrind is a GPL'd system for debugging and profiling Linux programs.
It can detect memory management and threading bugs.

Question:
What is Valgrind?"""
        }
    ],
    "stream": False,
    "keep_alive": -1,
    "options": {
        "temperature": 0,
        "num_ctx": 512,
        "num_predict": 14
    }
}

t0 = time.perf_counter()

print("Sending request...")
t1 = time.perf_counter()

response = requests.post(
    url,
    json=payload,
    timeout=30
)

t2 = time.perf_counter()

print("Response received.")

data = response.json()

t3 = time.perf_counter()

print("\n========== TIMING ==========")
print(f"HTTP request     : {(t2-t1)*1000:.2f} ms")
print(f"JSON parsing     : {(t3-t2)*1000:.2f} ms")
print(f"Total Python     : {(t3-t0)*1000:.2f} ms")

print("\n========== OLLAMA ==========")
print(f"Load             : {data.get('load_duration',0)/1e6:.2f} ms")
print(f"Prompt eval      : {data.get('prompt_eval_duration',0)/1e6:.2f} ms")
print(f"Prompt tokens    : {data.get('prompt_eval_count',0)}")
print(f"Generation       : {data.get('eval_duration',0)/1e6:.2f} ms")
print(f"Generated tokens : {data.get('eval_count',0)}")

print("\nAnswer:")
print(data["message"]["content"])

Sending request...
Response received.

========== TIMING ==========
HTTP request     : 891.95 ms
JSON parsing     : 0.73 ms
Total Python     : 892.92 ms

========== OLLAMA ==========
Load             : 487.03 ms
Prompt eval      : 34.10 ms
Prompt tokens    : 56
Generation       : 334.45 ms
Generated tokens : 14

Answer:
Valgrind is a debugging and profiling tool for Linux programs.


In [117]:
import time
from openai import OpenAI

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

MODEL_NAME = "qwen2.5:0.5b"

def generate_answer(question: str, context: str):

    clean_context = context[:250]
    print(clean_context)

    messages = [
        {
            "role": "system",
            "content": "Answer only from the context. Be concise."
        },
        {
            "role": "user",
            "content": f"{clean_context}\nQuestion: {question}"
        }
    ]

    t0 = time.perf_counter()

    try:
        response = ollama_client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0,
            max_tokens=100,
            stream=False,
            extra_body={
                "keep_alive": "10m",
                "options": {
                    "num_ctx": 512,
                    "num_predict": 20
                }
            }
        )

        generation_ms = (time.perf_counter() - t0) * 1000

        return {
            "answer": response.choices[0].message.content.strip(),
            "generation_ms": round(generation_ms, 2),
            "success": True
        }

    except Exception as e:
        return {
            "answer": None,
            "generation_ms": None,
            "success": False,
            "error": str(e)
        }

In [118]:
def answer_question(question: str):
    t_start = time.perf_counter()

    # 1. Retrieve
    retrieval = retrieve_hybrid(question)

    # 2. Guardrail: confidence check
    should_proceed, reason = check_retrieval_confidence(retrieval)
    if not should_proceed:
        return {
            "question": question,
            "answer": "I don't have enough information to answer that confidently.",
            "blocked": True,
            "block_reason": reason,
            "retrieval": retrieval,
            "total_ms": round((time.perf_counter() - t_start) * 1000, 2),
        }

    # 3. Build context + generate
    context = build_context_for_llm(retrieval)
    gen_result = generate_answer(question, context)

    if not gen_result["success"]:
        return {
            "question": question,
            "answer": "Sorry, I couldn't generate an answer right now.",
            "blocked": True,
            "block_reason": f"Generation failed: {gen_result['error']}",
            "retrieval": retrieval,
            "total_ms": round((time.perf_counter() - t_start) * 1000, 2),
        }

    t_end = time.perf_counter()

    return {
        "question": question,
        "answer": gen_result["answer"],
        "blocked": False,
        "block_reason": None,
        "retrieval": retrieval,
        "embedding_ms": retrieval["embedding_ms"],
        "qdrant_ms": retrieval["qdrant_ms"],
        "generation_ms": gen_result["generation_ms"],
        "total_ms": round((t_end - t_start) * 1000, 2),
    }

In [119]:
result = answer_question("what is valgrind")

print(f"Q: {result['question']}")
print(f"A: {result['answer']}")
print(f"\nBlocked: {result['blocked']}")
if result['blocked']:
    print(f"Reason: {result['block_reason']}")
print(f"\nTotal time: {result['total_ms']}ms")
if not result['blocked']:
    print(f"  embedding: {result['embedding_ms']}ms | qdrant: {result['qdrant_ms']}ms | generation: {result['generation_ms']}ms")

[Source 1]
About Valgrind. Valgrind is a GPL'd system for debugging and profiling Linux programs. With Valgrind's tool suite you can automatically detect many memory management and threading bugs, avoiding hours of frustrating bug-hunting, making you
Q: what is valgrind
A: Valgrind is a debugging and profiling tool for Linux programs.

Blocked: False

Total time: 3058.59ms
  embedding: 101.04ms | qdrant: 73.77ms | generation: 2883.21ms


In [41]:
models = groq_client.models.list()
for model in models.data:
    print(model.id)

qwen/qwen3.6-27b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
groq/compound
whisper-large-v3-turbo
openai/gpt-oss-20b
canopylabs/orpheus-arabic-saudi
groq/compound-mini
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
allam-2-7b
whisper-large-v3


In [22]:
!pip install elevenlabs -q


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from elevenlabs import ElevenLabs

ELEVENLABS_API_KEY = 
eleven_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)

VOICE_ID = # example default voice — pick one from your ElevenLabs dashboard

def text_to_speech(text: str, output_path: str = "answer.mp3"):
    t0 = time.perf_counter()
    audio = eleven_client.text_to_speech.convert(
        voice_id=VOICE_ID,
        text=text,
        model_id="eleven_flash_v2_5",   # fastest ElevenLabs model — good for latency-sensitive pipelines
    )
    with open(output_path, "wb") as f:
        for chunk in audio:
            f.write(chunk)
    t1 = time.perf_counter()
    return {"path": output_path, "tts_ms": round((t1 - t0) * 1000, 2)}

In [79]:
def answer_question_voice(question: str):
    result = answer_question(question)   # your existing retrieval + generation function

    if result["blocked"]:
        return result   # don't generate audio for blocked/refused answers

    tts_result = text_to_speech(result["answer"])
    result["audio_path"] = tts_result["path"]
    result["tts_ms"] = tts_result["tts_ms"]
    result["total_ms"] = round(result["total_ms"] + tts_result["tts_ms"], 2)

    return result

In [80]:
# test
final = answer_question_voice("what is valgrind")
print(f"A: {final['answer']}")
print(f"Audio saved to: {final['audio_path']}")
print(f"Total (retrieval + generation + TTS): {final['total_ms']}ms")

A: Valgrind is a GPL'd system for debugging and profiling Linux programs, helping detect memory management and threading bugs, improving program stability and performance.
Audio saved to: answer.mp3
Total (retrieval + generation + TTS): 1960.56ms


In [38]:
!pip install nest_asyncio -q


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import nest_asyncio
nest_asyncio.apply()

In [63]:
import asyncio
import websockets
import json
import base64
import time
import concurrent.futures

ELEVENLABS_WS_MODEL = "eleven_flash_v2_5"   # fastest, best for streaming

async def stream_llm_to_tts(question: str, context: str, voice_id: str, output_path: str = "answer_stream.mp3"):
    uri = f"wss://api.elevenlabs.io/v1/text-to-speech/{voice_id}/stream-input?model_id={ELEVENLABS_WS_MODEL}"

    t_start = time.perf_counter()
    first_audio_time = None
    audio_chunks = []
    full_text_parts = []

    async with websockets.connect(uri, additional_headers={"xi-api-key": ELEVENLABS_API_KEY}) as ws:
        await ws.send(json.dumps({
            "text": " ",
            "voice_settings": {"stability": 0.5, "similarity_boost": 0.8},
            "xi_api_key": ELEVENLABS_API_KEY,
        }))

        async def receive_audio():
            nonlocal first_audio_time
            async for message in ws:
                data = json.loads(message)
                if data.get("audio"):
                    if first_audio_time is None:
                        first_audio_time = time.perf_counter()
                    audio_chunks.append(base64.b64decode(data["audio"]))
                if data.get("isFinal"):
                    break

        receiver_task = asyncio.create_task(receive_audio())

        # --- run the BLOCKING Gemini stream in a background thread ---
        loop = asyncio.get_event_loop()
        executor = concurrent.futures.ThreadPoolExecutor()

        def run_gemini_stream():
            prompt = build_prompt_voice(question, context)
            response = model_llm.generate_content(prompt, stream=True)
            for chunk in response:
                if chunk.text:
                    full_text_parts.append(chunk.text)
                    # schedule the async send back on the event loop, thread-safely
                    asyncio.run_coroutine_threadsafe(
                        ws.send(json.dumps({"text": chunk.text})), loop
                    ).result()

        await loop.run_in_executor(executor, run_gemini_stream)

        await ws.send(json.dumps({"text": ""}))
        await receiver_task

    t_end = time.perf_counter()
    full_text = "".join(full_text_parts)

    with open(output_path, "wb") as f:
        for c in audio_chunks:
            f.write(c)

    return {
        "answer": full_text.strip(),
        "audio_path": output_path,
        "time_to_first_audio_ms": round((first_audio_time - t_start) * 1000, 2) if first_audio_time else None,
        "total_stream_ms": round((t_end - t_start) * 1000, 2),
    }

In [48]:
retrieval = retrieve_hybrid("what is valgrind")
context = build_context_for_llm(retrieval)

result = asyncio.run(stream_llm_to_tts("what is valgrind", context, VOICE_ID))

print(f"Answer: {result['answer']}")
print(f"Time to FIRST audio byte: {result['time_to_first_audio_ms']}ms")
print(f"Total stream time (full audio ready): {result['total_stream_ms']}ms")
print(f"Saved: {result['audio_path']}")

Answer: Valgrind is an instrumentation framework and multipurpose tool for debugging and profiling programs on Linux. It helps you automatically detect memory management and threading bugs, and it can also be used to build new dynamic analysis tools.
Time to FIRST audio byte: 3616.41ms
Total stream time (full audio ready): 4610.35ms
Saved: answer_stream.mp3


In [49]:
import time

t0 = time.perf_counter()
response = model_llm.generate_content(
    build_prompt_voice("what is valgrind", context),
    stream=True
)

for i, chunk in enumerate(response):
    t = time.perf_counter() - t0
    print(f"chunk {i} at {t*1000:.0f}ms: {repr(chunk.text[:50])}")

chunk 0 at 874ms: 'Valgr'
chunk 1 at 875ms: 'ind is a multipurpose tool for debugging and profi'
chunk 2 at 877ms: ' you detect memory management and threading bugs, '
chunk 3 at 877ms: ''


In [31]:
voices = eleven_client.voices.get_all()
for v in voices.voices:
    print(v.voice_id, "-", v.name, "-", v.category)

CwhRBWXzGAHq8TQ4Fs17 - Roger - Laid-Back, Casual, Resonant - premade
EXAVITQu4vr4xnSDxMaL - Sarah - Mature, Reassuring, Confident - premade
FGY2WhTYpPnrIDTdsKH5 - Laura - Enthusiast, Quirky Attitude - premade
IKne3meq5aSn9XLyUdCD - Charlie - Deep, Confident, Energetic - premade
JBFqnCBsd6RMkjVDRZzb - George - Warm, Captivating Storyteller - premade
N2lVS1w4EtoT3dr4eOWO - Callum - Husky Trickster - premade
SAz9YHcvj6GT2YYXdXww - River - Relaxed, Neutral, Informative - premade
SOYHLrjzK2X1ezoPC6cr - Harry - Fierce Warrior - premade
TX3LPaxmHKxFdv7VOQHJ - Liam - Energetic, Social Media Creator - premade
Xb7hH8MSUJpSbSDYk0k2 - Alice - Clear, Engaging Educator - premade
XrExE9yKIg1WjnnlVkGX - Matilda - Knowledgable, Professional - premade
bIHbv24MWmeRgasZH58o - Will - Relaxed Optimist - premade
cgSgspJ2msm6clMCkdW9 - Jessica - Playful, Bright, Warm - premade
cjVigY5qzO86Huf0OWal - Eric - Smooth, Trustworthy - premade
hpp4J3VqNfWAUOO0d1Us - Bella - Professional, Bright, Warm - premade
iP95p4

In [64]:
import logging
import time
import json
from datetime import datetime

# Configure logger
logger = logging.getLogger("rag_pipeline")
logger.setLevel(logging.INFO)

# avoid duplicate handlers if you re-run this cell
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
    handler.setFormatter(formatter)
    logger.addHandler(handler)

# also keep an in-memory log of every call, for later CSV export / P50 analysis
call_logs = []

In [65]:
def answer_question_voice_logged(question: str, request_id: str = None):
    request_id = request_id or f"req_{int(time.time()*1000)}"
    t_request_start = time.perf_counter()
    log_entry = {"request_id": request_id, "question": question, "timestamp": datetime.now().isoformat()}

    logger.info(f"[{request_id}] START | question='{question}'")

    # ---------------- Retrieval ----------------
    t0 = time.perf_counter()
    retrieval = retrieve_hybrid(question)
    t1 = time.perf_counter()
    retrieval_ms = round((t1 - t0) * 1000, 2)
    log_entry["retrieval_ms"] = retrieval_ms
    log_entry["top1_score"] = retrieval["top1_score"]
    log_entry["n_chunks_retrieved"] = len(retrieval["chunks"])
    logger.info(f"[{request_id}] RETRIEVAL done in {retrieval_ms}ms | top1_score={retrieval['top1_score']} | chunks={len(retrieval['chunks'])}")

    # ---------------- Guardrail: confidence check ----------------
    should_proceed, reason = check_retrieval_confidence(retrieval)
    log_entry["guardrail_passed"] = should_proceed
    log_entry["guardrail_reason"] = reason

    if not should_proceed:
        logger.warning(f"[{request_id}] BLOCKED at guardrail | reason='{reason}'")
        log_entry["blocked"] = True
        log_entry["answer"] = None
        log_entry["total_ms"] = round((time.perf_counter() - t_request_start) * 1000, 2)
        call_logs.append(log_entry)
        return log_entry

    logger.info(f"[{request_id}] GUARDRAIL passed")

    # ---------------- Generation ----------------
    context = build_context_for_llm(retrieval)
    t2 = time.perf_counter()
    gen_result = generate_answer(question, context)
    t3 = time.perf_counter()
    generation_ms = round((t3 - t2) * 1000, 2)
    log_entry["generation_ms"] = generation_ms

    if not gen_result["success"]:
        logger.error(f"[{request_id}] GENERATION FAILED after {generation_ms}ms | error={gen_result['error']}")
        log_entry["blocked"] = True
        log_entry["answer"] = None
        log_entry["error"] = gen_result["error"]
        log_entry["total_ms"] = round((time.perf_counter() - t_request_start) * 1000, 2)
        call_logs.append(log_entry)
        return log_entry

    logger.info(f"[{request_id}] GENERATION done in {generation_ms}ms | answer='{gen_result['answer'][:60]}...'")

    # ---------------- TTS ----------------
    t4 = time.perf_counter()
    try:
        tts_result = text_to_speech(gen_result["answer"], output_path=f"{request_id}.mp3")
        t5 = time.perf_counter()
        tts_ms = round((t5 - t4) * 1000, 2)
        log_entry["tts_ms"] = tts_ms
        log_entry["audio_path"] = tts_result["path"]
        logger.info(f"[{request_id}] TTS done in {tts_ms}ms | saved to {tts_result['path']}")
    except Exception as e:
        logger.error(f"[{request_id}] TTS FAILED | error={str(e)}")
        log_entry["tts_ms"] = None
        log_entry["audio_path"] = None
        log_entry["tts_error"] = str(e)

    # ---------------- Wrap up ----------------
    log_entry["blocked"] = False
    log_entry["answer"] = gen_result["answer"]
    total_ms = round((time.perf_counter() - t_request_start) * 1000, 2)
    log_entry["total_ms"] = total_ms

    logger.info(f"[{request_id}] COMPLETE | total={total_ms}ms "
                f"(retrieval={retrieval_ms}ms, generation={generation_ms}ms, tts={log_entry.get('tts_ms')}ms)")

    call_logs.append(log_entry)
    return log_entry

In [66]:
result = answer_question_voice_logged("what is valgrind")

12:37:20 | INFO | [req_1787036840144] START | question='what is valgrind'
12:37:20 | INFO | [req_1787036840144] RETRIEVAL done in 122.36ms | top1_score=0.8763 | chunks=5
12:37:20 | INFO | [req_1787036840144] GUARDRAIL passed
12:37:21 | INFO | [req_1787036840144] GENERATION done in 1705.61ms | answer='Valgrind is a multipurpose debugging and profiling system fo...'
12:37:24 | INFO | [req_1787036840144] TTS done in 2289.21ms | saved to req_1787036840144.mp3
12:37:24 | INFO | [req_1787036840144] COMPLETE | total=4126.36ms (retrieval=122.36ms, generation=1705.61ms, tts=2289.21ms)


In [67]:
# %% Latency benchmark — run logged pipeline across multiple test questions

test_questions = [
    "what is valgrind",
    "how were the first people to use the flute",
    "define assurance grade",
    "what is your body made of",
    "how to cash in a prepaid card",
    "what is the uber price per passenger",
    "are lions native in asia",
    "how long does a boil water notice last",
    "what job perks does a mechanical engineer need",
    "what regulation covers sop",
    # add more from your existing test_set for a realistic sample
]

print(f"Running {len(test_questions)} queries through full pipeline...\n")

for q in test_questions:
    answer_question_voice_logged(q)

print(f"\nCompleted {len(call_logs)} total calls.")

12:37:38 | INFO | [req_1787036858601] START | question='what is valgrind'
12:37:38 | INFO | [req_1787036858601] RETRIEVAL done in 94.84ms | top1_score=0.8763 | chunks=5
12:37:38 | INFO | [req_1787036858601] GUARDRAIL passed


Running 10 queries through full pipeline...



12:37:39 | INFO | [req_1787036858601] GENERATION done in 1057.15ms | answer='Valgrind is a multipurpose debugging and profiling system fo...'
12:37:40 | INFO | [req_1787036858601] TTS done in 1131.54ms | saved to req_1787036858601.mp3
12:37:40 | INFO | [req_1787036858601] COMPLETE | total=2290.4ms (retrieval=94.84ms, generation=1057.15ms, tts=1131.54ms)
12:37:40 | INFO | [req_1787036860894] START | question='how were the first people to use the flute'
12:37:41 | INFO | [req_1787036860894] RETRIEVAL done in 120.92ms | top1_score=0.75 | chunks=5
12:37:41 | INFO | [req_1787036860894] GUARDRAIL passed
12:37:42 | INFO | [req_1787036860894] GENERATION done in 1066.15ms | answer='It is hard to pinpoint the exact origins of the flute, but e...'
12:37:42 | INFO | [req_1787036860894] TTS done in 737.39ms | saved to req_1787036860894.mp3
12:37:42 | INFO | [req_1787036860894] COMPLETE | total=1934.79ms (retrieval=120.92ms, generation=1066.15ms, tts=737.39ms)
12:37:42 | INFO | [req_1787036862830] S


Completed 11 total calls.


In [68]:
import numpy as np
import pandas as pd

logs_df = pd.DataFrame(call_logs)
completed = logs_df[logs_df["blocked"] == False]   # exclude blocked/failed calls from latency stats

def pct(series, p):
    return round(np.percentile(series.dropna(), p), 2)

latency_report = pd.DataFrame([
    {
        "stage": "retrieval_ms",
        "P50": pct(completed["retrieval_ms"], 50),
        "P70": pct(completed["retrieval_ms"], 70),
        "P100": pct(completed["retrieval_ms"], 100),
        "mean": round(completed["retrieval_ms"].mean(), 2),
    },
    {
        "stage": "generation_ms",
        "P50": pct(completed["generation_ms"], 50),
        "P70": pct(completed["generation_ms"], 70),
        "P100": pct(completed["generation_ms"], 100),
        "mean": round(completed["generation_ms"].mean(), 2),
    },
    {
        "stage": "tts_ms",
        "P50": pct(completed["tts_ms"], 50),
        "P70": pct(completed["tts_ms"], 70),
        "P100": pct(completed["tts_ms"], 100),
        "mean": round(completed["tts_ms"].mean(), 2),
    },
    {
        "stage": "total_ms (end-to-end)",
        "P50": pct(completed["total_ms"], 50),
        "P70": pct(completed["total_ms"], 70),
        "P100": pct(completed["total_ms"], 100),
        "mean": round(completed["total_ms"].mean(), 2),
    },
])

print(latency_report.to_string(index=False))

logs_df.to_csv("pipeline_call_logs_full.csv", index=False)
latency_report.to_csv("latency_P50_P70_P100_report.csv", index=False)
print("\nSaved: pipeline_call_logs_full.csv, latency_P50_P70_P100_report.csv")

                stage     P50     P70    P100    mean
         retrieval_ms  108.91  122.36  136.37  107.53
        generation_ms 1037.30 1057.15 1705.61 1051.53
               tts_ms  749.35  990.56 2289.21  877.12
total_ms (end-to-end) 1911.31 2163.48 4126.36 2044.08

Saved: pipeline_call_logs_full.csv, latency_P50_P70_P100_report.csv


In [56]:
completed = logs_df[logs_df["blocked"] == False]
slowest = completed.nlargest(3, "generation_ms")
print(slowest[["request_id", "question", "retrieval_ms", "generation_ms", "tts_ms", "total_ms"]].to_string(index=False))


       request_id                               question  retrieval_ms  generation_ms  tts_ms  total_ms
req_1787035998406                       what is valgrind        115.17       16587.73 3222.28  19937.34
req_1787036028707 how long does a boil water notice last        211.20        1709.60  742.71   2671.53
req_1787036026547               are lions native in asia        109.62        1300.54  742.30   2158.68


In [57]:
retrieval = retrieve_hybrid("define assurance grade")
for i, c in enumerate(retrieval["chunks"], 1):
    print(f"{i}. score={c['fused_score']}")
    print(f"   {c['text'][:300]}\n")

1. score=0.9377
   Quality Assurance (QA) is defined by the College of American Pathologists as systematic monitoring of quality control results and quality practice parameters to assure that all systems are functioning in a manner appropriate to excellence in health care delivery.98 Quality assurance is a coordinated

2. score=0.855
   A graded approach to Quality Assurance derived from a. risk assessment process that assesses potential risk to: - nuclear safety. - peoples health and safety. - breach of site licence. - environmental or statutory requirements. - process losses. and determines the level of control that needs to be p

3. score=0.75
   Grading is a management process for determining the. degree of rigor, or effort (resources), which needs to be. applied to QA program implementation. Multiple factors. such as cost, schedule, environment, health and safety, mission, public perception, and security should be. considered and taken int

4. score=0.5341
   Quality Control and Q